# Gemini ve LangChain ile LLM API'larını Çağırma Giriş 🦜🔗

Bu notebook'ta LangChain aracılığıyla LLM API'larını nasıl kullanacağınızı öğreneceksiniz. Örnek olarak Google'ın Gemini API'sını kullanacağız. Bu notebook'un sonunda, LangChain kullanarak API çağrıları yapmayı ve bunu neden yaptığımızı bileceksiniz.

## ⚙️ Kurulum

👉 Kurulum aşamasında oluşturduğumuz `.env` dosyasındaki ortam değişkenlerini yüklemek için aşağıdaki hücreyi çalıştırın:

In [1]:
from dotenv import load_dotenv

load_dotenv() # Load environment variables from .env file

True

👉 Hücrenin çıktısı "`True`" mu? Harika! Artık Gemini API ile kimlik doğrulaması yapmak için kullanılacak bir `GOOGLE_API_KEY` ortam değişkeni kurmuş olduk.

Eğer değilse, yardım isteyin.

## Basit Bir API Çağrısı Yapma

Bu notebook'ta şunların nasıl yapılacağını göstereceğiz:
1. Google'ın kendi kütüphanesini kullanarak API çağrısı yapma.
2. Aynı işlemi LangChain kullanarak yapma.

## Google Generative AI Kütüphanesini Kullanma

In [2]:
from google import genai

In [3]:
client = genai.Client()

response = client.models.generate_content(
    model="gemini-2.5-flash-lite",
    contents="What is the capital of France?",
)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


`response` nesnesine bir göz atalım.

In [4]:
response.candidates[0].content.parts[0].text

'The capital of France is **Paris**.'

Gerçek cevabı nasıl alabileceğinizi görüyor musunuz?

Neyse ki, cevabı hemen almak için sadece `.text` özelliğini kullanabiliriz. Deneyin.

In [5]:
response.text

'The capital of France is **Paris**.'

Gemini cevaplarını Markdown formatında döndürür. Bunu kullanalım!

In [6]:
from IPython.display import Markdown
Markdown(response.text)

The capital of France is **Paris**.

Oluşturma parametrelerini de değiştirebilirsiniz. `google.genai` kullanarak bunu şu şekilde yaparsınız:

In [7]:
from google import genai
from google.genai import types # We need to import types for the config

client = genai.Client()

response = client.models.generate_content(
    model="gemini-2.5-flash-lite",
    contents="Write a social media post about how much you're learning about transformers.",
    config=types.GenerateContentConfig(
        max_output_tokens=200,
        temperature=1.0
    )
)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [8]:
Markdown(response.text)

Here are a few options for a social media post about learning transformers, ranging in tone and focus. Choose the one that best fits your style!

**Option 1: Enthusiastic & General**

> Mind officially blown! 🤯 Diving deep into the world of **Transformers** lately and I'm just in awe of how these models work. The attention mechanisms, the parallel processing... it's like unlocking a whole new level of AI understanding. So much to learn, so exciting! #Transformers #MachineLearning #AI #DeepLearning #NLP

**Option 2: Slightly More Technical & Focused on the "Aha!" Moment**

> Seriously getting my head around **Transformers** and the elegance of self-attention is just *chef's kiss*. ✨ It’s the key to so much of what we see in advanced NLP today. Feels like a significant step in my AI journey! Anyone else obsessed with this architecture? #TransformerModels #AIResearch #NaturalLanguage

In [10]:
import os

Harika. Ancak başka bir API denemek istediğinizi düşünün, örneğin OpenAI'nin veya Anthropic'in?

Onların dokümantasyonlarını incelemek ve tüm kodunuzu onların API'sini kullanacak şekilde yeniden yazmak zorunda kalırsınız. Tabii ki benzer olacaktır, ancak aynı olmayacaktır.

Neyse ki LangChain var!

## LangChain Kullanma 🦜🔗

Neden LangChain kullanırsınız?

1. **Model-Bağımsız Kod**

   LangChain, farklı LLM sağlayıcıları (Google, OpenAI, Anthropic, vb.) arasında minimal kod değişikliği ile geçiş yapmanızı sağlayan soyutlamalar sunar. Google API'sine doğrudan kod yazarsanız, sağlayıcı değiştirmek önemli ölçüde yeniden düzenleme gerektirir.

2. **Birleşik Arayüz**

   LangChain, altta yatan API'den bağımsız olarak farklı LLM sağlayıcıları arasında etkileşimleri standartlaştırır ve tutarlı yöntemler ile yanıt formatları sunar.

3. **Bileşenlerle Çalışabilirlik**

   LangChain'in zincir ve pipeline mimarisi, tüm alt yapıyı kendiniz halletmeden prompt, bellek ve erişim sistemlerini birleştiren karmaşık iş akışları oluşturmayı kolaylaştırır.

4. **Yerleşik Araçlar**

   LangChain, çıktı ayrıştırma, prompt şablonları ve kendiniz uygulamanız gereken diğer yardımcı araçları içerir.

[LangChain'in chat entegrasyonları listesi](https://docs.langchain.com/oss/python/integrations/chat)'ne gidin ve entegrasyon listesine bakın. Favori LLM sağlayıcınızı bulabiliyor musunuz?

Kodumuzda `chat_models.ChatGoogleGenerativeAI` kullanmak istemiyoruz çünkü bu özellikle Gemini için yapılmış. LLM'yi değiştirmek istersek, modeli başlatma şeklimizi değiştirmek zorunda kalırız. Neyse ki LangChain bir modeli başlatmak için daha genel bir yol sunar.

Gemini'yi tekrar kullanalım, ancak şimdi LangChain'in genel Chat Models'ini kullanarak.

👉 [LangChain'in "Models" dokümantasyonu](https://docs.langchain.com/oss/python/langchain/models) sayfasına gidin ve Gemini kullanarak bir chat modelinin nasıl başlatılacağını bulun.

İpuçları:
1. Hemen "Basic Usage" bölümüne gidin.
2. Kullanmak istediğiniz modeli seçerek doğru dokümantasyonu hemen görebilirsiniz.

In [11]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    google_api_key=os.getenv("GOOGLE_API_KEY")
)

response = llm.invoke("What is the capital of France?")

print(response.content)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


The capital of France is **Paris**.


Modelin en temel kullanımı sadece `.invoke()` metodunu kullanmaktır:

In [12]:
from pprint import pprint

In [13]:
pprint(llm.__dict__)

{'_serialized': {'id': ['langchain_google_genai',
                        'chat_models',
                        'ChatGoogleGenerativeAI'],
                 'kwargs': {'default_metadata': [],
                            'google_api_key': {'id': ['GOOGLE_API_KEY'],
                                               'lc': 1,
                                               'type': 'secret'},
                            'location': None,
                            'max_retries': 6,
                            'model': 'gemini-2.5-flash-lite',
                            'n': 1,
                            'output_version': None,
                            'temperature': 0.7},
                 'lc': 1,
                 'name': 'ChatGoogleGenerativeAI',
                 'type': 'constructor'},
 '_use_vertexai': False,
 'additional_headers': None,
 'base_url': None,
 'cache': None,
 'cached_content': None,
 'callbacks': None,
 'client': <google.genai.client.Client object at 0x000001D7ADEBF3B0>,


Yanıta bir göz atalım. Nesnenin tüm öznitelik ve metodlarını içeren `__dict__`'ini güzel şekilde yazdırmak için `pprint()` kullanıyoruz.

In [14]:
from pprint import pprint
pprint(response.__dict__)

{'additional_kwargs': {},
 'content': 'The capital of France is **Paris**.',
 'id': 'lc_run--019dede5-e8ca-7962-a6d9-d9f55621a79e-0',
 'invalid_tool_calls': [],
 'name': None,
 'response_metadata': {'finish_reason': 'STOP',
                       'model_name': 'gemini-2.5-flash-lite',
                       'model_provider': 'google_genai',
                       'safety_ratings': []},
 'tool_calls': [],
 'type': 'ai',
 'usage_metadata': {'input_token_details': {'cache_read': 0},
                    'input_tokens': 8,
                    'output_tokens': 8,
                    'total_tokens': 16}}


Cevabı çıkarın ve görüntüleyin. Markdown formatında olduğunu unutmayın, bu yüzden güzel görünmesini sağlayabilirsiniz.

In [15]:
from IPython.display import Markdown

Markdown(response.content)

The capital of France is **Paris**.

Modelin temperature değerini `.temperature` özniteliğine erişerek kontrol edebilirsiniz. Deneyin:

In [16]:
print(llm.temperature)

0.7


Modeli kullanmadan önce, özniteliklere yeni değerler atayarak oluşturma parametrelerini de ayarlayabiliriz.

Daha önce Google'ın kütüphanesini kullanarak sosyal medya gönderisi yazmak için yaptığımızın eşdeğerini kodlamaya çalışın.

> _Not_: Normal olarak modelin `max_output_tokens` değerini ayarlayabilmemiz gerekir (modeli başlatırken veya daha sonra özniteliği değiştirerek). _langchain_google_genai_'nin mevcut sürümü (4.1.1) bir [hataya](https://github.com/langchain-ai/langchain-google/issues/1454) sahip ve bu çalışmıyor. Geçici çözüm? `max_output_tokens`'ı `.invoke()` metodunun bir parametresi olarak ayarlayın.

In [17]:
# Set the maximum number of output tokens to 200
max_tokens = 200

# Set the temperature to 1.0
llm.temperature = 1.0

# Generate a response with the new settings
response = llm.invoke(
    "Write an enthusiastic social media post about learning transformers.",
    max_output_tokens=max_tokens
)

# Display the response
from IPython.display import Markdown
Markdown(response.content)

## 🚀 Brace Yourselves! I'm Diving Headfirst into the MAGICAL World of TRANSFORMERS! 🤯

Seriously, I'm SO hyped about this! Forget everything you thought you knew about AI – because **TRANSFORMERS** are here to blow your mind! 🤯

I've been dipping my toes in, and let me tell you, it's like unlocking a whole new level of understanding for language, code, and so much more. 🤩 These neural networks are absolute GENIUSES at understanding context and relationships within data, and it's making my brain do a happy dance! 💃🧠

From generating mind-bending text to powering the coolest AI applications I've seen, transformers are truly the engine of innovation right now. I'm so excited to get my hands dirty, build some awesome things, and see what kind of creative magic I can conjure up! ✨

If you're even a little bit curious about the future of

Bunun avantajı? Bu LangChain Chat Model birçok başka API'yi destekleyebilir.

Başka bir modele geçmek için değiştirmeniz gereken tek şeyler:
1. Diğer model için bir API anahtarı alın ve kodunuzda tanımlayın.
2. Modeli başlatırken model ve sağlayıcıyı değiştirin.

### Çoklu Mesajlar

`.invoke()` fonksiyonunu sadece tek bir mesajla kullanmak biraz kısıtlayıcı.

Şu gibi birden fazla mesaj sağlayabilirsiniz:
- `SystemMessage` veya sistem mesajları: modelin nasıl davranacağını söylemek için
- `HumanMessage` veya Kullanıcı mesajları: kullanıcıdan gelen girdi
- `AIMessage` veya Asistan mesajları: modelden gelen yanıt

Bir sosyal medya yazarı yapalım.

Modele nasıl davranacağını açıklayan bir sistem mesajı göndereceğiz. Sonra kullanıcı mesajında, kendimizi sadece yazacağı konuyu vermekle sınırlayabiliriz.

Bunu nasıl yapacağınızı öğrenmek için [LangChain'in "Messages" dokümantasyonu](https://docs.langchain.com/oss/python/langchain/messages)'na bakın.

Sistem mesajı için ilhama mı ihtiyacınız var? İşte başlamanız için temel bir talimat:

```python
"""Sen Üretken AI öğrencisi için gönderiler yazan yaratıcı bir sosyal medya yazarısın.
Gönderilerinde her zaman kelime oyunu ve harekete geçirici çağrı bulunur.
Gönderilerin maksimum 200 karakter uzunluğundadır.
Her zaman emoji kullanırsın.
"""
```

In [19]:
from langchain_core.messages import SystemMessage, HumanMessage
from IPython.display import Markdown

messages = [
    SystemMessage(content="""
Sen üretken AI öğrencileri için sosyal medya gönderileri yazan yaratıcı bir yazarsın.
Gönderilerinde maksimum 200 karakter kullanırsın.
Her zaman emoji kullanırsın.
Harekete geçirici bir çağrı içerir.
"""),

    HumanMessage(content="Write a post about learning transformers.")
]

response = llm.invoke(messages)

Markdown(response.content)

Transformatorları öğrenmek mi? 🤩 Bu heyecan verici teknolojiyle yapay zekayı bir sonraki seviyeye taşıyın! 🚀 Devam etmek için bu heyecan verici dünyaya dalın. #Transformers #AI #MachineLearning #DeepLearning

🏁 Tebrikler! Artık LangChain kullanarak çoklu mesajlarla temel prompt yazma konusunda uzmanlaştınız.